# זרימת עבודה עם מעורבות אנושית במסגרת Microsoft Agent Framework

## 🎯 מטרות למידה

במחברת זו, תלמד כיצד ליישם זרימות עבודה **עם מעורבות אנושית** באמצעות `RequestInfoExecutor` במסגרת Microsoft Agent Framework. תבנית עוצמתית זו מאפשרת לך לעצור זרימות עבודה של בינה מלאכותית כדי לאסוף קלט אנושי, ולהפוך את הסוכנים שלך לאינטרקטיביים תוך מתן שליטה לבני אדם על החלטות קריטיות.

## 🔄 מהי מעורבות אנושית בזרימה?

**מעורבות אנושית בזרימה (HITL)** היא תבנית עיצוב שבה סוכני AI עוצרים את הביצוע כדי לבקש קלט אנושי לפני שהם ממשיכים. זה חיוני עבור:

- ✅ **החלטות קריטיות** - לקבל אישור אנושי לפני ביצוע פעולות חשובות
- ✅ **מצבים מעורפלים** - לאפשר לבני אדם להבהיר כש-AI אינו בטוח
- ✅ **העדפות משתמש** - לשאול משתמשים לבחור בין אפשרויות שונות
- ✅ **ציות ובטיחות** - להבטיח פיקוח אנושי על פעולות מפוקחות
- ✅ **חוויות אינטראקטיביות** - לבנות סוכנים שיחה שמגיבים לקלט משתמש

## 🏗️ כיצד זה פועל במסגרת Microsoft Agent Framework

המסגרת מספקת שלושה מרכיבים מרכזיים ל-HITL:

1. **`RequestInfoExecutor`** - מבצע מיוחד שעוצר את זרימת העבודה ומשדר אירוע `RequestInfoEvent`
2. **`RequestInfoMessage`** - מחלקת בסיס עבור מטעני בקשה ממוקדים הנשלחים לבני אדם
3. **`RequestResponse`** - מקשר תגובות אנושיות עם בקשות מקוריות באמצעות `request_id`

**תבנית זרימת עבודה:**
```
Agent detects need for input
    ↓
Sends message to RequestInfoExecutor
    ↓
Workflow pauses & emits RequestInfoEvent
    ↓
Application collects human input (console, UI, etc.)
    ↓
Application sends RequestResponse via send_responses_streaming()
    ↓
Workflow resumes with human input
```

## 🏨 הדוגמה שלנו: הזמנת מלון עם אישור משתמש

נבנה על זרימת העבודה המותנית על ידי הוספת אישור אנושי **לפני** הצעת יעדים חלופיים:

1. המשתמש מבקש יעד (למשל, "פריז")
2. `availability_agent` בודק אם יש חדרים פנויים
3. **אם אין חדרים** → `confirmation_agent` שואל "האם תרצה לראות חלופות?"
4. זרימת העבודה **עוצרת** באמצעות `RequestInfoExecutor`
5. **האנוש מגיב** "כן" או "לא" דרך קלט קונסול
6. `decision_manager` מנתב לפי התשובה:
   - **כן** → הצגת יעדים חלופיים
   - **לא** → ביטול בקשת ההזמנה
7. הצגת התוצאה הסופית

זה מדגים כיצד לתת למשתמשים שליטה על ההצעות של הסוכן!

---

בואו נתחיל! 🚀


## שלב 1: ייבוא הספריות הנדרשות

אנו מייבאים את רכיבי מסגרת הסוכן הסטנדרטיים יחד עם **מחלקות ייחודיות לאדם בתהליך**:
- `RequestInfoExecutor` - מבצע שעוצר את זרימת העבודה לקבלת קלט אנושי
- `RequestInfoEvent` - אירוע שנפלט כאשר מתבקש קלט אנושי
- `RequestInfoMessage` - מחלקת בסיס לטעונים בקשות מטיפוסים ספציפיים
- `RequestResponse` - מקשר תגובות אנושיות עם בקשות
- `WorkflowOutputEvent` - אירוע לזיהוי פלטים מזרימת עבודה


In [ ]:
import asyncio
import json
import os
from dataclasses import dataclass
from typing import Annotated, Any, Never

from agent_framework import (
    AgentExecutor,
    AgentExecutorRequest,
    AgentExecutorResponse,
    ChatMessage,
    Executor,
    RequestInfoEvent,          # NEW: Event when human input is requested
    RequestInfoExecutor,       # NEW: Executor that gathers human input
    RequestInfoMessage,        # NEW: Base class for request payloads
    RequestResponse,           # NEW: Correlates response with request
    Role,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowOutputEvent,       # NEW: Event for workflow outputs
    WorkflowRunState,          # NEW: Enum of workflow run states
    WorkflowStatusEvent,       # NEW: Event for run state changes
    ai_function,
    executor,
    handler,                   # NEW: Decorator for executor methods
)

# 🤖 Azure OpenAI client integration
from agent_framework.openai import OpenAIChatClient
from dotenv import load_dotenv
from IPython.display import HTML, display
from pydantic import BaseModel

print("✅ All imports successful!")
print("🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse")

✅ All imports successful!
🔄 Human-in-the-loop components loaded: RequestInfoExecutor, RequestInfoEvent, RequestResponse


## שלב 2: הגדרת מודלים בפיידנטיק לפלטים מבניים

מודלים אלו מגדירים את **הסקימה** שהסוכנים יחזירו. אנו שומרים על כל המודלים מזרימת העבודה המותנית ומוסיפים:

**חדש בלולאת האדם:**
- `HumanFeedbackRequest` - תת-מחלקה של `RequestInfoMessage` שמגדירה את מטען הבקשה שנשלח לאנשים
  - מכיל `prompt` (שאלה לשאול) ו-`destination` (קונטקסט על העיר שאינה זמינה)


In [22]:
# Existing models from conditional workflow
class BookingCheckResult(BaseModel):
    """Result from checking hotel availability at a destination."""
    destination: str
    has_availability: bool
    message: str


class AlternativeResult(BaseModel):
    """Suggested alternative destination when no rooms available."""
    alternative_destination: str
    reason: str


class BookingConfirmation(BaseModel):
    """Booking suggestion when rooms are available."""
    destination: str
    action: str
    message: str


# NEW: Pydantic model for agent's response format
class ConfirmationQuestion(BaseModel):
    """
    Pydantic model used by confirmation_agent's response_format.
    This is what the agent will output as JSON.
    """
    question: str  # The question to ask the user
    destination: str  # The unavailable destination for context


# NEW: Dataclass for RequestInfoExecutor
@dataclass
class HumanFeedbackRequest(RequestInfoMessage):
    """
    Request sent to RequestInfoExecutor asking if user wants alternatives.
    
    MUST be a dataclass subclassing RequestInfoMessage for type compatibility.
    This is what gets sent to the RequestInfoExecutor.
    """
    prompt: str = ""  # The question to ask the user
    destination: str = ""  # The unavailable destination for context


print("✅ Pydantic models defined:")
print("   - BookingCheckResult (availability check)")
print("   - AlternativeResult (alternative suggestion)")
print("   - BookingConfirmation (booking confirmation)")
print("   - ConfirmationQuestion (agent response format) 🆕")
print("   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕")

✅ Pydantic models defined:
   - BookingCheckResult (availability check)
   - AlternativeResult (alternative suggestion)
   - BookingConfirmation (booking confirmation)
   - ConfirmationQuestion (agent response format) 🆕
   - HumanFeedbackRequest (RequestInfoMessage for HITL) 🆕


## שלב 3: צור את כלי הזמנת המלונות

אותו כלי מהזרימה המותנית - בודק אם יש חדרים זמינים ביעד.


In [23]:
@ai_function(description="Check hotel room availability for a destination city")
def hotel_booking(destination: Annotated[str, "The destination city to check for hotel rooms"]) -> str:
    """
    Simulates checking hotel room availability.
    
    Returns JSON string with availability status.
    """
    display(
        HTML(f"""
        <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
            <strong>🔍 Tool Invoked:</strong> hotel_booking("{destination}")
        </div>
    """)
    )

    # Simulate availability check
    cities_with_rooms = ["stockholm", "seattle", "tokyo", "london", "amsterdam"]
    has_rooms = destination.lower() in cities_with_rooms

    result = {"has_availability": has_rooms, "destination": destination}

    return json.dumps(result)


print("✅ hotel_booking tool created with @ai_function decorator")

✅ hotel_booking tool created with @ai_function decorator


## שלב 4: הגדרת פונקציות תנאי לניתוב

אנו זקוקים ל**ארבע פונקציות תנאי** עבור זרימת העבודה עם מעורבות אדם:

**מזרימת עבודה מותנית:**
1. `has_availability_condition` - מנווט כאשר בתי המלון זמינים
2. `no_availability_condition` - מנווט כאשר בתי המלון אינם זמינים

**חדשות עבור מעורבות אדם:**
3. `user_wants_alternatives_condition` - מנווט כאשר המשתמש אומר "כן" לאלטרנטיבות
4. `user_declines_alternatives_condition` - מנווט כאשר המשתמש אומר "לא" לאלטרנטיבות


In [24]:
# Existing condition functions from conditional workflow
def has_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels ARE available."""
    if not isinstance(message, AgentExecutorResponse):
        return True

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                <strong>✅ Condition Check:</strong> has_availability = <strong>{result.has_availability}</strong> for {result.destination}
            </div>
        """)
        )
        return result.has_availability
    except Exception as e:
        display(HTML(f"""<div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'><strong>⚠️  Error:</strong> {str(e)}</div>"""))
        return False


def no_availability_condition(message: Any) -> bool:
    """Condition for routing when hotels are NOT available."""
    if not isinstance(message, AgentExecutorResponse):
        return False

    try:
        result = BookingCheckResult.model_validate_json(message.agent_run_response.text)
        display(
            HTML(f"""
            <div style='padding: 12px; background: #ffecb3; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                <strong>❌ Condition Check:</strong> no_availability for {result.destination}
            </div>
        """)
        )
        return not result.has_availability
    except Exception as e:
        return False


# NEW: Condition functions for human-in-the-loop routing
def user_wants_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user WANTS to see alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            wants_alternatives = "wants to see alternative" in msg_text or "want to see alternative" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
                    <strong>🔍 User Decision:</strong> User wants alternatives = <strong>{wants_alternatives}</strong>
                </div>
            """)
            )
            
            return wants_alternatives
    
    return False
def user_declines_alternatives_condition(message: Any) -> bool:
    """
    Condition for routing when user DECLINES alternatives.
    
    Checks the AgentExecutorRequest sent by decision_manager.
    """
    # Check if it's an AgentExecutorRequest (what decision_manager sends)
    if isinstance(message, AgentExecutorRequest):
        # Check the message text to determine user's choice
        if message.messages and len(message.messages) > 0:
            msg_text = message.messages[0].text.lower()
            declined = "declined" in msg_text or "has declined" in msg_text
            
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fce4ec; border-left: 4px solid #c2185b; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 User Decision:</strong> User declined alternatives = <strong>{declined}</strong>
                </div>
            """)
            )
            
            return declined
    
    return False
print("✅ Condition functions defined:")
print("   - has_availability_condition (routes when rooms exist)")
print("   - no_availability_condition (routes when no rooms)")
print("   - user_wants_alternatives_condition (routes when user says yes) 🆕")
print("   - user_declines_alternatives_condition (routes when user says no) 🆕")

✅ Condition functions defined:
   - has_availability_condition (routes when rooms exist)
   - no_availability_condition (routes when no rooms)
   - user_wants_alternatives_condition (routes when user says yes) 🆕
   - user_declines_alternatives_condition (routes when user says no) 🆕


## שלב 5: צור את מבצע מנהל ההחלטות

זהו **המרכיב המרכזי בתבנית האדם בלולאה**! `DecisionManager` הוא `Executor` מותאם אישית ש-

1. **מקבל משוב מהאדם** דרך אובייקטים מסוג `RequestResponse`
2. **מעבד את החלטת המשתמש** (כן/לא)
3. **מנווט את זרימת העבודה** על ידי שליחת הודעות לסוכנים המתאימים

תכונות עיקריות:
- משתמש בדקורטור `@handler` כדי לחשוף שיטות כשלבי זרימת עבודה
- מקבל `RequestResponse[HumanFeedbackRequest, str]` שמכיל גם את הבקשה המקורית וגם את תשובת המשתמש
- מפיק הודעות פשוטות "כן" או "לא" שמפעילות את פונקציות התנאי שלנו


In [25]:
class DecisionManager(Executor):
    """
    Coordinates workflow routing based on human feedback.
    
    This executor receives RequestResponse objects from the RequestInfoExecutor
    and makes routing decisions by sending simple messages that trigger
    condition functions.
    """

    def __init__(self, id: str | None = None):
        super().__init__(id=id or "decision_manager")

    @handler
    async def on_human_feedback(
        self,
        feedback: RequestResponse[HumanFeedbackRequest, str],
        ctx: WorkflowContext[AgentExecutorRequest],
    ) -> None:
        """
        Process human feedback and let the workflow route based on conditions.
        
        The RequestResponse contains:
        - feedback.data: The user's string reply (e.g., "yes" or "no")
        - feedback.original_request: The HumanFeedbackRequest with context
        
        This handler just displays feedback and passes the RequestResponse through.
        The routing is done by condition functions on the edges.
        """
        user_reply = (feedback.data or "").strip().lower()
        destination = getattr(feedback.original_request, "destination", "unknown")

        display(
            HTML(f"""
            <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
                <strong>🎯 Decision Manager:</strong> Processing user reply: <strong>"{user_reply}"</strong> for {destination}
            </div>
        """)
        )

        if user_reply == "yes":
            display(
                HTML("""
                <div style='padding: 12px; background: #c8e6c9; border-left: 4px solid #4caf50; border-radius: 4px; margin: 10px 0;'>
                    <strong>➡️  Routing:</strong> User wants alternatives → Will route to alternative_agent
                </div>
            """)
            )
            # Create and send a message for the alternative_agent
            user_msg = ChatMessage(
                Role.USER,
                text=f"The user wants to see alternative destinations near {destination}. Please suggest one.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        elif user_reply == "no":
            display(
                HTML("""
                <div style='padding: 12px; background: #ffcdd2; border-left: 4px solid #f44336; border-radius: 4px; margin: 10px 0;'>
                    <strong>🚫 Routing:</strong> User declined alternatives → Will route to cancellation_agent
                </div>
            """)
            )
            # Create and send a message for the cancellation_agent
            user_msg = ChatMessage(
                Role.USER,
                text="The user has declined to see alternatives. Please acknowledge their decision.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))
        
        else:
            # Handle unexpected input - treat as decline
            display(
                HTML(f"""
                <div style='padding: 12px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 4px; margin: 10px 0;'>
                    <strong>⚠️  Warning:</strong> Unexpected input "{user_reply}" - treating as decline
                </div>
            """)
            )
            user_msg = ChatMessage(
                Role.USER,
                text="The user has declined to see alternatives. Please acknowledge their decision.",
            )
            await ctx.send_message(AgentExecutorRequest(messages=[user_msg], should_respond=True))


print("✅ DecisionManager executor created with @handler method for human feedback")

✅ DecisionManager executor created with @handler method for human feedback


## שלב 6: צור מבצע הצגה מותאם אישית

אותו מבצע הצגה מהזרימה התנאי - מפיק תוצאות סופיות כפלט של הזרימה.


In [26]:
@executor(id="prepare_human_request")
async def prepare_human_request(
    response: AgentExecutorResponse, 
    ctx: WorkflowContext[HumanFeedbackRequest]
) -> None:
    """
    Transform agent response into HumanFeedbackRequest for RequestInfoExecutor.
    
    This executor bridges the type gap between:
    - confirmation_agent outputs AgentExecutorResponse with ConfirmationQuestion JSON
    - request_info_executor expects HumanFeedbackRequest (RequestInfoMessage dataclass)
    """
    display(
        HTML("""
        <div style='padding: 12px; background: #e1f5fe; border-left: 4px solid #0288d1; border-radius: 4px; margin: 10px 0;'>
            <strong>🔄 Transform:</strong> Converting ConfirmationQuestion to HumanFeedbackRequest
        </div>
    """)
    )
    
    # Parse the agent's Pydantic output (ConfirmationQuestion)
    confirmation = ConfirmationQuestion.model_validate_json(response.agent_run_response.text)
    
    # Convert to HumanFeedbackRequest dataclass for RequestInfoExecutor
    feedback_request = HumanFeedbackRequest(
        prompt=confirmation.question,
        destination=confirmation.destination
    )
    
    # Send the properly typed RequestInfoMessage to the RequestInfoExecutor
    await ctx.send_message(feedback_request)


@executor(id="display_result")
async def display_result(response: AgentExecutorResponse, ctx: WorkflowContext[Never, str]) -> None:
    """
    Display the final result as workflow output.
    
    This executor receives the final agent response and yields it as the workflow output.
    """
    display(
        HTML("""
        <div style='padding: 15px; background: #f3e5f5; border-left: 4px solid #9c27b0; border-radius: 4px; margin: 10px 0;'>
            <strong>📤 Display Executor:</strong> Yielding workflow output
        </div>
    """)
    )

    await ctx.yield_output(response.agent_run_response.text)


print("✅ prepare_human_request executor created with @executor decorator")
print("✅ display_result executor created with @executor decorator")

✅ prepare_human_request executor created with @executor decorator
✅ display_result executor created with @executor decorator


## שלב 7: טעינת משתני סביבה

הגדר את לקוח ה-LLM (Microsoft Foundry / Azure OpenAI).


In [ ]:
# Load environment variables
load_dotenv()

from azure.identity import AzureCliCredential

# Azure OpenAI via the Responses API. Sign in with `az login` for keyless Entra ID auth.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API,
# so this sample calls Azure OpenAI directly. OpenAIChatClient uses the Responses API.
chat_client = OpenAIChatClient(
    azure_endpoint=os.environ["AZURE_OPENAI_ENDPOINT"],
    credential=AzureCliCredential(),
    model_id=os.environ["AZURE_OPENAI_DEPLOYMENT"],
)

print("✅ Chat client configured with Azure OpenAI (Responses API)")


✅ Chat client configured with GitHub Models


## שלב 8: יצירת סוכני בינה מלאכותית ומבצעים

אנו יוצרים **שש רכיבי תהליך עבודה**:

**סוכנים (עטופים ב-AgentExecutor):**
1. **availability_agent** - בודק זמינות במלון באמצעות הכלי
2. **confirmation_agent** - 🆕 מכין את בקשת האישור האנושית
3. **alternative_agent** - מציע ערים חלופיות (כאשר המשתמש אומר כן)
4. **booking_agent** - מעודד הזמנה (כאשר חדרים זמינים)
5. **cancellation_agent** - 🆕 מטפל בהודעת הביטול (כאשר המשתמש אומר לא)

**מבצעים מיוחדים:**
6. **request_info_executor** - 🆕 `RequestInfoExecutor` שמשהה את תהליך העבודה לקבלת קלט אנושי
7. **decision_manager** - 🆕 מבצע מותאם אישית שמפנה לפי תגובת האדם (כבר מוגדר למעלה)


In [ ]:
# Agent 1: Check availability with tool (same as conditional workflow)
availability_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a hotel booking assistant that checks room availability. "
            "Use the hotel_booking tool to check if rooms are available at the destination. "
            "Return JSON with fields: destination (string), has_availability (bool), and message (string). "
            "The message should summarize the availability status."
        ),
        tools=[hotel_booking],
        response_format=BookingCheckResult,
    ),
    id="availability_agent",
)

# Agent 2: NEW - Prepare human confirmation request
confirmation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user's requested destination has no available hotel rooms. "
            "Create a polite message asking if they would like to see alternative destinations nearby. "
            "Return a JSON with: destination (the unavailable city), and question (a friendly yes/no question). "
            "Keep the question concise and friendly."
        ),
        response_format=ConfirmationQuestion,  # Use Pydantic model for agent output
    ),
    id="confirmation_agent",
)

# Agent 3: Suggest alternative (when user says yes)
alternative_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful travel assistant. When a user cannot find hotels in their requested city, "
            "suggest an alternative nearby city that has availability. "
            "Return JSON with fields: alternative_destination (string) and reason (string). "
            "Make your suggestion sound appealing and helpful."
        ),
        response_format=AlternativeResult,
    ),
    id="alternative_agent",
)

# Agent 4: Suggest booking (when rooms available)
booking_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a booking assistant. The user has found available hotel rooms. "
            "Encourage them to book by highlighting the destination's appeal. "
            "Return JSON with fields: destination (string), action (string), and message (string). "
            "The action should be 'book_now' and message should be encouraging."
        ),
        response_format=BookingConfirmation,
    ),
    id="booking_agent",
)

# Agent 5: NEW - Handle cancellation when user declines alternatives
class CancellationMessage(BaseModel):
    """Message when user declines alternatives."""
    status: str
    message: str

cancellation_agent = AgentExecutor(
    chat_client.as_agent(
        instructions=(
            "You are a helpful assistant. The user has declined to see alternative hotel destinations. "
            "Create a polite cancellation message. "
            "Return JSON with: status (should be 'cancelled'), and message (a friendly acknowledgment). "
            "Keep the message brief and understanding."
        ),
        response_format=CancellationMessage,
    ),
    id="cancellation_agent",
)

# NEW: RequestInfoExecutor - pauses workflow to gather human input
request_info_executor = RequestInfoExecutor(id="request_info")

# NEW: DecisionManager instance - routes based on human feedback
decision_manager = DecisionManager(id="decision_manager")

display(
    HTML("""
    <div style='padding: 15px; background: #e3f2fd; border-left: 4px solid #2196f3; border-radius: 4px; margin: 10px 0;'>
        <strong>✅ Created Workflow Components:</strong>
        <ul style='margin: 10px 0 0 0;'>
            <li><strong>availability_agent</strong> - Checks availability with hotel_booking tool</li>
            <li><strong>confirmation_agent</strong> 🆕 - Prepares human confirmation request</li>
            <li><strong>alternative_agent</strong> - Suggests alternative cities</li>
            <li><strong>booking_agent</strong> - Encourages booking</li>
            <li><strong>cancellation_agent</strong> 🆕 - Handles user declining alternatives</li>
            <li><strong>request_info_executor</strong> 🆕 - Pauses workflow for human input</li>
            <li><strong>decision_manager</strong> 🆕 - Routes based on human response</li>
        </ul>
    </div>
""")
)


## שלב 9: בניית זרימת העבודה עם אדם בלולאה

כעת אנו בונים את גרף זרימת העבודה עם **ניווט מותנה** הכולל את מסלול האדם בלולאה:

**מבנה זרימת העבודה:**
```
availability_agent (START)
        ↓
   Evaluate conditions
        ↙                    ↘
[no_availability]        [has_availability]
        ↓                        ↓
confirmation_agent          booking_agent
        ↓                        ↓
prepare_human_request      display_result
        ↓
request_info_executor (PAUSE)
        ↓
decision_manager
   ↙         ↘
[yes]        [no]
   ↓           ↓
alternative  cancellation
   ↓           ↓
display_result
```

**קצוות מפתח:**
- `availability_agent → confirmation_agent` (כאשר אין חדרים)
- `confirmation_agent → prepare_human_request` (שינוי סוג)
- `prepare_human_request → request_info_executor` (השהיה עבור האדם)
- `request_info_executor → decision_manager` (תמיד - מספק RequestResponse)
- `decision_manager → alternative_agent` (כאשר המשתמש אומר "כן")
- `decision_manager → cancellation_agent` (כאשר המשתמש אומר "לא")
- `availability_agent → booking_agent` (כאשר יש חדרים זמינים)
- כל המסלולים מסתיימים ב- `display_result`


In [29]:
# Build the workflow with human-in-the-loop routing
workflow = (
    WorkflowBuilder()
    .set_start_executor(availability_agent)
    
    # NO AVAILABILITY PATH (with human-in-the-loop)
    .add_edge(availability_agent, confirmation_agent, condition=no_availability_condition)
    .add_edge(confirmation_agent, prepare_human_request)  # Transform to HumanFeedbackRequest
    .add_edge(prepare_human_request, request_info_executor)  # Send to RequestInfoExecutor
    .add_edge(request_info_executor, decision_manager)    # Always goes to decision manager
    
    # Decision manager routes based on user response
    .add_edge(decision_manager, alternative_agent, condition=user_wants_alternatives_condition)
    .add_edge(decision_manager, cancellation_agent, condition=user_declines_alternatives_condition)
    .add_edge(alternative_agent, display_result)
    .add_edge(cancellation_agent, display_result)
    
    # HAS AVAILABILITY PATH (no human input needed)
    .add_edge(availability_agent, booking_agent, condition=has_availability_condition)
    .add_edge(booking_agent, display_result)
    
    .build()
)

display(
    HTML("""
    <div style='padding: 20px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); color: white; border-radius: 8px; margin: 10px 0;'>
        <h3 style='margin: 0 0 15px 0;'>✅ Workflow Built Successfully!</h3>
        <p style='margin: 0; line-height: 1.6;'>
            <strong>Human-in-the-Loop Routing:</strong><br>
            • If <strong>NO availability</strong> → confirmation_agent → prepare_human_request → request_info_executor → <strong>PAUSE FOR HUMAN</strong> → decision_manager<br>
            &nbsp;&nbsp;• If user says <strong>YES</strong> → alternative_agent → display_result<br>
            &nbsp;&nbsp;• If user says <strong>NO</strong> → cancellation_agent → display_result<br>
            • If <strong>availability</strong> → booking_agent → display_result (no human input needed)
        </p>
    </div>
""")
)

## שלב 10: הרץ מקרה בדיקה 1 - עיר ללא זמינות (פריז עם אישור אנושי)

במבחן זה מודגם **המחזור המלא של האדם בלולאה**:

1. בקש מלון בפריז
2. סוכן הזמינות בודק → אין חדרים
3. סוכן האישור יוצר שאלה המופנית לאדם
4. מבצע בקשת המידע **עוצר את זרימת העבודה** ומשגר את `RequestInfoEvent`
5. **היישום מזהה את האירוע ומציע למשתמש בקונסולה**
6. המשתמש מקליד "כן" או "לא"
7. היישום שולח תגובה דרך `send_responses_streaming()`
8. מנהל ההחלטות מנווט על בסיס התגובה
9. התוצאה הסופית מוצגת

**דפוס מפתח:**
- השתמש ב`workflow.run_stream()` לאיטרציה הראשונה
- השתמש ב`workflow.send_responses_streaming(pending_responses)` לאיטרציות הבאות
- האזן ל`RequestInfoEvent` כדי לזהות מתי נדרש קלט אנושי
- האזן ל`WorkflowOutputEvent` כדי ללכוד תוצאות סופיות


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #e65100;'>🧪 TEST CASE 1: Paris (No Availability - Human-in-the-Loop)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → confirmation_agent → request_info_executor → <strong>PAUSE</strong> → decision_manager → (depends on user input)</p>
    </div>
""")
)

# Create request for Paris
request_paris = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Paris")], 
    should_respond=True
)

# Human-in-the-loop execution pattern
pending_responses: dict[str, str] | None = None
completed = False
workflow_output: str | None = None

print("\n🔄 Starting human-in-the-loop workflow...")
print("=" * 60)

while not completed:
    # First iteration uses run_stream with the request
    # Subsequent iterations use send_responses_streaming with collected human responses
    if pending_responses:
        print(f"\n📤 Sending human responses: {pending_responses}")
        stream = workflow.send_responses_streaming(pending_responses)
        pending_responses = None  # Clear immediately after sending
    else:
        print(f"\n🚀 Starting workflow with request: 'I want to book a hotel in Paris'")
        stream = workflow.run_stream(request_paris)
    
    # Collect all events from this iteration
    events = [event async for event in stream]
    
    # Process events
    requests: list[tuple[str, str]] = []  # (request_id, prompt)
    
    for event in events:
        # Check for human input requests
        if isinstance(event, RequestInfoEvent) and isinstance(event.data, HumanFeedbackRequest):
            print(f"\n⏸️  WORKFLOW PAUSED - Human input requested!")
            print(f"   Request ID: {event.request_id}")
            print(f"   Destination: {event.data.destination}")
            requests.append((event.request_id, event.data.prompt))
        
        # Check for workflow outputs
        elif isinstance(event, WorkflowOutputEvent):
            workflow_output = str(event.data)
            completed = True
            print(f"\n✅ Workflow completed with output!")
    
    # If we have human requests, prompt the user
    if requests and not completed:
        responses: dict[str, str] = {}
        for req_id, prompt in requests:
            print(f"\n{'='*60}")
            print(f"💬 QUESTION FOR YOU:")
            print(f"   {prompt}")
            print(f"{'='*60}")
            
            # Get user input (in notebook, this will pause execution)
            answer = input("👉 Enter 'yes' or 'no': ").strip().lower()
            
            print(f"\n📝 You answered: {answer}")
            responses[req_id] = answer
        
        pending_responses = responses

print(f"\n{'='*60}")
print(f"🏆 FINAL WORKFLOW OUTPUT:")
print(f"{'='*60}")

# Display final result
if workflow_output:
    # Try to parse as JSON for pretty display
    try:
        result_data = json.loads(workflow_output)
        if "alternative_destination" in result_data:
            result_obj = AlternativeResult.model_validate_json(workflow_output)
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #FFD700 0%, #FFA500 100%); border-radius: 12px; box-shadow: 0 4px 12px rgba(255,165,0,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0; color: #333;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> ✅ Accepted alternatives</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Alternative Suggestion:</strong> 🏨 {result_obj.alternative_destination}</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Reason:</strong> {result_obj.reason}</p>
                    </div>
                </div>
            """)
            )
        else:
            # User declined
            display(
                HTML(f"""
                <div style='padding: 25px; background: linear-gradient(135deg, #f44336 0%, #e91e63 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(244,67,54,0.3); margin: 20px 0;'>
                    <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT</h3>
                    <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ❌ No rooms in Paris</p>
                        <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>User Decision:</strong> 🚫 Declined alternatives</p>
                        <p style='margin: 0; font-size: 14px; color: #666;'><strong>Result:</strong> Booking request cancelled</p>
                    </div>
                </div>
            """)
            )
    except:
        print(workflow_output)


🔄 Starting human-in-the-loop workflow...

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: 032c8fce-b9d1-400e-ba8d-afd2248e2926
   Destination: Paris

💬 QUESTION FOR YOU:
   Unfortunately, there are no rooms available in Paris. Would you like to explore nearby alternative destinations?

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: yes

📤 Sending human responses: {'032c8fce-b9d1-400e-ba8d-afd2248e2926': 'yes'}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'



⏸️  WORKFLOW PAUSED - Human input requested!
   Request ID: cf48dad0-ee5e-4f60-8806-341a7a292bd4
   Destination: Paris

💬 QUESTION FOR YOU:
   I'm sorry to inform you that there are no available hotel rooms in Paris. Would you like me to suggest nearby alternative destinations?

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'

📝 You answered: 

📤 Sending human responses: {'cf48dad0-ee5e-4f60-8806-341a7a292bd4': ''}

🚀 Starting workflow with request: 'I want to book a hotel in Paris'


## שלב 11: הרץ מקרה מבחן 2 - עיר עם זמינות (סטוקהולם - ללא צורך בקלט אנושי)

מבחן זה מדגים את **הנתיב הישיר** כאשר חדרים זמינים:

1. בקשת מלון בסטוקהולם
2. availability_agent בודק → חדרים זמינים ✅
3. booking_agent מציע הזמנה
4. display_result מציג אישור
5. **אין צורך בקלט אנושי!**

תהליך העבודה חוסך את מסלול האדם בלולאה לחלוטין כאשר חדרים זמינים.


In [ ]:
display(
    HTML("""
    <div style='padding: 20px; background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 8px; margin: 20px 0;'>
        <h3 style='margin: 0 0 10px 0; color: #1b5e20;'>🧪 TEST CASE 2: Stockholm (Has Availability - No Human Input)</h3>
        <p style='margin: 0;'>Expected workflow path: availability_agent → booking_agent → display_result (direct, no pause)</p>
    </div>
""")
)

# Create request for Stockholm
request_stockholm = AgentExecutorRequest(
    messages=[ChatMessage(Role.USER, text="I want to book a hotel in Stockholm")], 
    should_respond=True
)

# Run the workflow (should complete without human input)
events_stockholm = await workflow.run(request_stockholm)
outputs_stockholm = events_stockholm.get_outputs()

# Display results
if outputs_stockholm:
    result_stockholm = BookingConfirmation.model_validate_json(outputs_stockholm[0])

    display(
        HTML(f"""
        <div style='padding: 25px; background: linear-gradient(135deg, #4caf50 0%, #8bc34a 100%); color: white; border-radius: 12px; box-shadow: 0 4px 12px rgba(76,175,80,0.3); margin: 20px 0;'>
            <h3 style='margin: 0 0 15px 0;'>🏆 WORKFLOW RESULT (Stockholm - No Human Input)</h3>
            <div style='background: white; color: #333; padding: 20px; border-radius: 8px;'>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Status:</strong> ✅ Rooms Available!</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Destination:</strong> 🏨 {result_stockholm.destination}</p>
                <p style='margin: 0 0 10px 0; font-size: 16px;'><strong>Action:</strong> {result_stockholm.action}</p>
                <p style='margin: 0 0 10px 0; font-size: 14px; color: #666;'><strong>Message:</strong> {result_stockholm.message}</p>
                <p style='margin: 10px 0 0 0; font-size: 12px; color: #999; font-style: italic;'>Note: No human input was requested because rooms were available!</p>
            </div>
        </div>
    """)
    )

## נקודות מפתח ופרקטיקות מיטביות לאנוש-בלולאה

### ✅ מה שלמדת:

#### 1. **תבנית RequestInfoExecutor**
תבנית האנוש-בלולאה במסגרת Microsoft Agent משתמשת בשלושה מרכיבים מרכזיים:
- `RequestInfoExecutor` - עוצר את זרימת העבודה ומשדר אירועים
- `RequestInfoMessage` - מחלקה בסיסית לטעינת בקשות מקודדות (יש להרחיב ממנה!)
- `RequestResponse` - מקשר תגובות אנושיות לבקשות המקוריות

**הבנה קריטית:**
- `RequestInfoExecutor` אינו אוסף קלט בעצמו - הוא רק עוצר את זרימת העבודה
- קוד האפליקציה שלך חייב להאזין ל-`RequestInfoEvent` ולאסוף קלט
- עליך לקרוא ל-`send_responses_streaming()` עם מילון שממפה `request_id` לתשובת המשתמש

#### 2. **תבנית ביצוע בזרם**
```python
# איטרציה ראשונה
stream = workflow.run_stream(initial_request)

# איטרציות הבאות (לאחר קלט אנושי)
stream = workflow.send_responses_streaming(pending_responses)

# תמיד לעבד אירועים
events = [event async for event in stream]
```

#### 3. **ארכיטקטורת מונעת אירועים**
האזן לאירועים ספציפיים כדי לשלוט בזרימת העבודה:
- `RequestInfoEvent` - נדרש קלט אנושי (זרימת העבודה נעצרה)
- `WorkflowOutputEvent` - התוצאה הסופית זמינה (זרימת העבודה הושלמה)
- `WorkflowStatusEvent` - שינויים במצב (IN_PROGRESS, IDLE_WITH_PENDING_REQUESTS וכו')

#### 4. **מבצעים מותאמים עם @handler**
ה-`DecisionManager` ממחיש כיצד ליצור מבצעים ש:
- משתמש בעיטור `@handler` כדי לחשוף שיטות כשלבי זרימת עבודה
- מקבל הודעות מתוחכמות (למשל, `RequestResponse[HumanFeedbackRequest, str]`)
- מנתב זרימת עבודה על ידי שליחת הודעות למבצעים אחרים
- ניגש להקשר דרך `WorkflowContext`

#### 5. **ניתוב מותנה עם החלטות אנושיות**
ניתן ליצור פונקציות תנאי שמעריכות תגובות אנושיות:
```python
def user_wants_alternatives_condition(message: Any) -> bool:
    response_text = message.agent_run_response.text.lower()
    return response_text == "yes"
```

### 🎯 יישומים בעולם האמיתי:

1. **זרימות עבודה לאישור**
   - לקבל אישור מנהל לפני עיבוד דוחות הוצאות
   - לדרוש ביקורת אנושית לפני שליחת מיילים אוטומטיים
   - לאשר עסקאות בעלות ערך גבוה לפני ביצוען

2. **בקרה על תוכן**
   - לסמן תוכן חשוד לסקירה אנושית
   - לבקש מהמארגנים לקבל החלטות סופיות במקרים מורכבים
   - להעביר לאנושיים כאשר הביטחון של ה-AI נמוך

3. **שירות לקוחות**
   - לאפשר ל-AI לטפל בשאלות שגרתיות באופן אוטומטי
   - להעביר נושאים מורכבים לסוכני אנוש
   - לשאול את הלקוח אם הוא רוצה לדבר עם אדם

4. **עיבוד נתונים**
   - לבקש מהאנשים לפתור הזנות נתונים דו-משמעותיות
   - לאשר פרשנויות AI למסמכים לא ברורים
   - לאפשר למשתמשים לבחור בין פרשנויות תקפות רבות

5. **מערכות קריטיות לבטיחות**
   - לדרוש אישור אנושי לפני פעולות בלתי הפיכות
   - לקבל אישור לפני גישה לנתונים רגישים
   - לאשר החלטות בתעשיות מפוקחות (בריאות, פיננסים)

6. **סוכנים אינטראקטיביים**
   - לבנות בוטים שיחתמו עם שאלות המשך
   - ליצור מדריכים שמנחים משתמשים בתהליכים מורכבים
   - לעצב סוכנים שמשתפים פעולה עם אנשים צעד-אחר-צעד

### 🔄 השוואה: עם מול ללא אנוש-בלולאה

| תכונה | זרימת עבודה מותנית | זרימת עבודה באנוש-בלולאה |
|---------|---------------------|---------------------------|
| **ביצוע** | `workflow.run()` יחיד | לולאה עם `run_stream()` + `send_responses_streaming()` |
| **קלט משתמש** | אין (אוטומטי לחלוטין) | קלט אינטראקטיבי דרך `input()` או UI |
| **מרכיבים** | סוכנים + מבצעים | + RequestInfoExecutor + DecisionManager |
| **אירועים** | רק AgentExecutorResponse | RequestInfoEvent, WorkflowOutputEvent וכו' |
| **עצירה** | ללא עצירה | זרימת העבודה נעצרת ב-RequestInfoExecutor |
| **בקרת אנוש** | ללא בקרה אנושית | אנשים מקבלים החלטות מפתח |
| **מקרה שימוש** | אוטומציה | שיתוף פעולה ופיקוח |

### 🚀 תבניות מתקדמות:

#### נקודות החלטה אנושיות מרובות
ניתן לכלול מספר צמתים `RequestInfoExecutor` באותה זרימת עבודה:
```python
.add_edge(agent1, request_info_1)  # החלטה אנושית ראשונה
.add_edge(decision_manager_1, agent2)
.add_edge(agent2, request_info_2)  # החלטה אנושית שנייה
.add_edge(decision_manager_2, final_agent)
```

#### טיפול ב-Timeout
יישום הגבלת זמן לתגובות אנושיות:
```python
import asyncio

try:
    answer = await asyncio.wait_for(
        asyncio.to_thread(input, "Enter yes/no: "),
        timeout=60.0
    )
except asyncio.TimeoutError:
    answer = "no"  # ברירת מחדל לאפשרות בטוחה
```

#### אינטגרציה עשירה לממשק משתמש
במקום `input()`, השתלב עם ממשקי רשת, Slack, Teams וכו':
```python
if isinstance(event, RequestInfoEvent):
    # שלח הודעה לערוץ המועדף על המשתמש
    await slack_client.send_message(
        user_id=current_user,
        text=event.data.prompt,
        request_id=event.request_id
    )
```

#### אנוש-בלולאה מותנית
יש לבקש קלט אנושי רק במצבים ספציפיים:
```python
def needs_human_approval_condition(message: Any) -> bool:
    # הפנה לאדם רק אם הביטחון נמוך או הערך גבוה
    if result.confidence < 0.7 or result.value > 10000:
        return True
    return False
```

### ⚠️ פרקטיקות מיטביות:

1. **תמיד להרחיב את RequestInfoMessage**
   - מספק בטיחות טיפוס ואימות
   - מאפשר הקשר עשיר להצגת UI
   - מבהיר כוונה לכל סוג בקשה

2. **השתמש בהנחיות תיאוריות**
   - כלול הקשר לגבי מה שאתה שואל
   - הסבר השלכות כל בחירה
   - שמור על שאלות פשוטות וברורות

3. **טפל בקלט לא צפוי**
   - אמת תגובות משתמש
   - ספק ברירות מחדל לקלט לא חוקי
   - הצג הודעות שגיאה ברורות

4. **עקוב אחרי מזהי בקשה**
   - השתמש בקורלציה בין request_id לתגובות
   - אל תנסה לנהל מצב באופן ידני

5. **תכנן ללא חסימת תהליכים**
   - אל תחסום תהליכים בהמתנה לקלט
   - השתמש בתבניות אסינכרוניות בכל מקום
   - תמוך במופעי זרימת עבודה מקבילים

### 📚 מושגים קשורים:

- **דפדפן סוכנים** - חטוף קריאות סוכן (פנקס קודם)
- **ניהול מצב זרימת עבודה** - שימור מצב בין הרצות
- **שיתוף פעולה בין סוכנים** - שילוב אנוש-בלולאה עם צוותי סוכנים
- **ארכיטקטורות מונעות אירועים** - בניית מערכות תגובתיות עם אירועים

---

### 🎓 מזל טוב!

שלטת בזרימות עבודה עם אנוש-בלולאה במסגרת Microsoft Agent! אתה עכשיו יודע איך:
- ✅ לעצור זרימות עבודה לאיסוף קלט אנושי
- ✅ להשתמש ב-RequestInfoExecutor ו-RequestInfoMessage
- ✅ לנהל ביצוע זרם עם אירועים
- ✅ ליצור מבצעים מותאמים עם @handler
- ✅ לנווט זרימות עבודה על פי החלטות אנושיות
- ✅ לבנות סוכני AI אינטראקטיביים שמשתפים פעולה עם אנשים

**זוהי תבנית קריטית לבניית מערכות AI אמינות ובעלות בקרה!** 🚀


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**כתב ויתור**:
מסמך זה תורגם באמצעות שירות תרגום אוטומטי [Co-op Translator](https://github.com/Azure/co-op-translator). למרות שאנו שואפים לדיוק, יש לקחת בחשבון שתרגומים אוטומטיים עלולים להכיל שגיאות או אי-דיוקים. יש להחשיב את המסמך המקורי בשפתו הטבעית כמקור הסמכות. למידע קריטי מומלץ להשתמש בתרגום מקצועי על ידי מתרגם אדם. אנו לא אחראים לכל אי-הבנה או פירוש שגוי הנובע מהשימוש בתרגום זה.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
